# 07_自訓練模型推論.py
**電腦視覺與深度學習 › Day20 YOLO11n 物件偵測(預訓練到自己訓練)**

這支程式自動找出最新訓練好的 best.pt,接上鏡頭即時偵測,讓你看到自己的模型真的能用。

> 🎧 有逐段旁白、逐句拆解的版本在 [pyl Academy](https://friencdking0000.github.io/pyl-academy/#/learn/PY-CVAI/7/8)。
>
> **怎麼用**:由上往下,每格按左邊 ▶ 執行;或選單「執行階段 › 全部執行」。
>
> ⚡ 這支會用到深度學習,建議先到「執行階段 › 變更執行階段類型」選 **T4 GPU**。
>
> 📷 Colab 沒有攝影機、也開不了視窗:下面會**改用範例圖片**代替鏡頭,影像直接顯示在格子底下(只顯示前幾張)。

In [ ]:
#@title 0. 準備環境(下載課程範例檔、安裝套件)
!git clone -q --depth 1 https://github.com/friencdking0000/2026AI-STUST.git /content/2026AI-STUST 2>/dev/null; echo 課程檔案已就緒
%cd "/content/2026AI-STUST/Week2/DAY10"
!pip -q install ultralytics
import sys, os; sys.argv = sys.argv[:1]  # 讓 argparse / sys.argv 在 notebook 裡也正常
__file__ = os.path.join(os.getcwd(), '07_自訓練模型推論.py')  # notebook 沒有 __file__,程式靠它找同資料夾的檔案

In [ ]:
#@title 修正 data.yaml 路徑(檔案裡寫的是講師電腦的 C:/ 路徑)
import re, pathlib
for y in pathlib.Path('datasets').rglob('data.yaml'):
    t = y.read_text(encoding='utf-8')
    y.write_text(re.sub(r'(?m)^path:.*$', 'path: ' + y.parent.resolve().as_posix(), t), encoding='utf-8')
    print('已修正', y)

### 還沒訓練過?
這支要用 05 訓練出來的結果(`runs/train/`)。Colab 每個 notebook 的檔案不共用,可以執行下一格,在這裡先訓練一次(建議開 T4 GPU,20 回合約 5～10 分鐘)。

In [ ]:
#@title (選用)先在這裡跑一次 05 訓練
!printf '\n\n' | python "05_訓練自己的YOLO.py"

### 需要先訓練出 best.pt
這支會找 05 訓練出來的 `best.pt`。Colab 每個 notebook 的檔案不共用:可以先在同一個執行階段跑 05,或用下一格上傳你電腦裡的 `best.pt`。

In [ ]:
#@title 上傳 best.pt(放到 runs/train/weights/)
import os, shutil
from google.colab import files
up = files.upload()
os.makedirs('runs/train/weights', exist_ok=True)
for n in up: shutil.move(n, 'runs/train/weights/best.pt')

In [ ]:
#@title 1. Colab 沒有攝影機和視窗:改用範例圖片、影像直接顯示在下面
import cv2, numpy as np, urllib.request, os
from google.colab.patches import cv2_imshow
def _pyl_sample():
    if not os.path.exists('/content/_sample.jpg'):
        urllib.request.urlretrieve('https://ultralytics.com/images/bus.jpg', '/content/_sample.jpg')
    return cv2.imread('/content/_sample.jpg')
class _PylCam:
    def __init__(self, *a, **k): self.n = 0; self.img = _pyl_sample()
    def isOpened(self): return True
    def read(self):
        self.n += 1
        return (True, self.img.copy()) if self.n <= 30 else (False, None)
    def set(self, *a): return True
    def get(self, *a): return 0
    def release(self): pass
_pyl_VC = cv2.VideoCapture
cv2.VideoCapture = lambda src=0, *a, **k: _PylCam() if isinstance(src, int) else _pyl_VC(src, *a, **k)
_pyl_shown = [0]
def _pyl_imshow(name, img):
    _pyl_shown[0] += 1
    if _pyl_shown[0] <= 6:
        print('▼', name); cv2_imshow(img)
cv2.imshow = _pyl_imshow
cv2.waitKey = lambda *a: ord('q') if _pyl_shown[0] >= 6 else -1
cv2.destroyAllWindows = lambda *a: None
cv2.namedWindow = lambda *a, **k: None
cv2.createTrackbar = lambda *a, **k: None
cv2.getTrackbarPos = lambda *a, **k: 0
cv2.setMouseCallback = lambda *a, **k: None
cv2.resizeWindow = cv2.moveWindow = cv2.setWindowProperty = cv2.setWindowTitle = lambda *a, **k: None
_pyl_imread = cv2.imread
def _pyl_imread2(path, *a, **k):
    img = _pyl_imread(path, *a, **k)
    if img is None and not os.path.exists(str(path)):
        print('(找不到', path, ':課程 repo 沒附這張圖,改用範例圖片)')
        img = _pyl_sample()
    return img
cv2.imread = _pyl_imread2
print('已改用範例圖片代替攝影機')

### 前置與匯入
前提是已經跑過 05,產生了 best.pt。這支程式會自動找最新的那個。

In [ ]:
# 07 — 用剛訓練好的 YOLO 模型做 webcam 即時推論
#
# 前置：跑過 05_訓練自己的YOLO.py，會產生 runs/train/weights/best.pt
# 這支腳本自動找最新一個 run 的 best.pt

import os
import time
from pathlib import Path

import cv2
import numpy as np
from ultralytics import YOLO
from PIL import Image, ImageDraw, ImageFont

### 找最新的 best.pt
在兩個位置用 rglob 找所有 best.pt,依修改時間排序,拿最新的。找不到就提醒你先訓練。載入後印出你的類別名稱。

In [ ]:
BASE = Path(__file__).parent

# ====== 找最新一個 best.pt ======
# 05 現在會用絕對路徑存到 DAY10/runs/；為了相容以前跑到 project root 的版本，也掃那邊
search_dirs = [BASE / "runs", BASE.parent.parent / "runs"]
weights_candidates = []
for d in search_dirs:
    if d.exists():
        weights_candidates.extend(d.rglob("best.pt"))
weights_candidates = sorted(weights_candidates, key=lambda p: p.stat().st_mtime, reverse=True)
if not weights_candidates:
    raise FileNotFoundError(
        "找不到 best.pt。先跑 05_訓練自己的YOLO.py 完成訓練"
    )
MODEL_PATH = weights_candidates[0]
print(f"載入最新模型：{MODEL_PATH}")
if len(weights_candidates) > 1:
    print(f"（共找到 {len(weights_candidates)} 個 best.pt，用最新的）")

model = YOLO(str(MODEL_PATH))

### 字型與畫標籤
如果你的類別名稱是中文,就需要中文字型。配色和畫標籤的函式和 02 一樣。

In [ ]:
print(f"類別名稱：{model.names}")

# ====== 中文字型（自己訓練的類別名如果是中文才用得上）======
FONT_CANDIDATES = [
    "C:/Windows/Fonts/msjh.ttc",
    "C:/Windows/Fonts/msyh.ttc",
    "C:/Windows/Fonts/simhei.ttf",
]
FONT_PATH = next((p for p in FONT_CANDIDATES if os.path.exists(p)), None)


def 依類別配色(name):
    h = hash(name) & 0xFFFFFF
    return (h & 0xFF, (h >> 8) & 0xFF, (h >> 16) & 0xFF)

### 開鏡頭
打開鏡頭,提示按 q 離開、按 s 存圖。

In [ ]:
def 畫中文標籤(img_bgr, text, xy, size, bg_bgr):
    img_pil = Image.fromarray(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    draw = ImageDraw.Draw(img_pil)
    font = ImageFont.truetype(FONT_PATH, size)
    l, t, r, b = draw.textbbox((0, 0), text, font=font)
    tw, th = r - l, b - t
    x, y = xy
    bg_rgb = (bg_bgr[2], bg_bgr[1], bg_bgr[0])
    draw.rectangle((x, y, x + tw + 6, y + th + 6), fill=bg_rgb)
    draw.text((x + 3, y + 3), text, font=font, fill=(255, 255, 255))
    return cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)


# ====== 開 webcam ======
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("無法開啟 webcam")

print("按 q 離開；按 s 存目前畫面到 snapshot.png")

fps_hist = []

### 推論
和 02 一樣一行 predict。conf 是信心門檻,自己訓練的模型如果常常亂框,可以調高一點。

### 畫出每個框
取出每個框的座標、信心和類別,畫框和標籤。這次類別名稱直接用你訓練時取的名字。

### FPS 與模型名稱
左上角顯示平均 FPS,以及這次用的是哪一個訓練資料夾的模型。

### 顯示、存圖、收拾
顯示畫面;按 q 離開,按 s 把目前畫面存成 snapshot.png。

In [ ]:
while True:
    ret, frame = cap.read()
    if not ret:
        break
    t0 = time.time()

    results = model.predict(frame, imgsz=640, conf=0.4, verbose=False)

    for r in results:
        boxes = r.boxes
        if boxes is None:
            continue
        for i in range(len(boxes)):
            x1, y1, x2, y2 = boxes.xyxy[i].tolist()
            conf = float(boxes.conf[i])
            cls_id = int(boxes.cls[i])
            name  = model.names[cls_id]
            color = 依類別配色(name)
            cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), color, 3)
            frame = 畫中文標籤(frame, f"{name} {conf*100:.0f}%",
                              (int(x1), max(0, int(y1) - 30)), 20, color)

    fps_hist.append(1.0 / max(time.time() - t0, 1e-3))
    if len(fps_hist) > 30:
        fps_hist.pop(0)
    fps = sum(fps_hist) / len(fps_hist)
    frame = 畫中文標籤(frame, f"FPS {fps:.1f}  |  {MODEL_PATH.parent.parent.name}",
                     (10, 10), 22, (30, 30, 30))

    cv2.imshow("YOLO 自訓練模型", frame)
    key = cv2.waitKey(1) & 0xFF
    if key == ord("q"):
        break
    if key == ord("s"):
        cv2.imwrite("snapshot.png", frame)
        print("已存 snapshot.png")

cap.release()
cv2.destroyAllWindows()

## 小結
你的模型在真實鏡頭上跑起來了。下一支 08 會把同樣的畫面串流到瀏覽器,讓別人也看得到。

👉 回到 [pyl Academy 這堂課](https://friencdking0000.github.io/pyl-academy/#/learn/PY-CVAI/7/8) 看逐句拆解、做練習。